# Semana 19 — Fine-Tuning com PEFT/LoRA
**Fase 4: Parameter-Efficient Fine-Tuning (PEFT)** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/04-Phase-4-Fine-Tuning/notebooks/semana_19_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-19-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/04-Phase-4-Fine-Tuning/docs/Semana-19-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_19_gabarito.ipynb`.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
!pip -q install peft

In [ ]:
import torch, json, random, os, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0); print(device)
MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
ALVOS = ["q_proj", "k_proj", "v_proj", "o_proj"]

## Passo 1 — Dataset (o da Semana 17)

In [ ]:
# --- Dados brutos: chamados de TI fictícios, com a sujeira típica de dados reais ---
import random
SISTEMA_TICKETS = ('Você é um classificador de chamados de TI. Responda apenas com um JSON com as chaves '
                   '"categoria", "prioridade" e "resumo".')
CATEGORIAS = ["acesso", "hardware", "software", "rede", "financeiro"]; PRIORIDADES = ["baixa", "media", "alta"]
def gerar_tickets_brutos(n=600, seed=0):
    rnd = random.Random(seed)
    modelos = {
        "acesso": [("Não consigo entrar no {x}, aparece senha inválida.", "Falha de login no {x}"),
                   ("Meu usuário foi bloqueado no {x} depois de várias tentativas.", "Usuário bloqueado no {x}")],
        "hardware": [("O {x} não liga desde ontem.", "{x} não liga"), ("A tela do {x} está piscando sem parar.", "Tela do {x} com defeito")],
        "software": [("O {x} fecha sozinho quando abro um arquivo grande.", "{x} fecha inesperadamente"),
                     ("Preciso instalar o {x} na minha máquina.", "Instalação do {x}")],
        "rede": [("A internet do {x} está muito lenta.", "Lentidão de rede no {x}"), ("O wifi do {x} cai a cada dez minutos.", "Queda de wifi no {x}")],
        "financeiro": [("Minha nota de reembolso de {x} reais não foi paga.", "Reembolso de {x} reais pendente"),
                       ("Fui cobrado em duplicidade em {x} reais na fatura.", "Cobrança duplicada de {x} reais")],
    }
    slots = {"acesso": ["SAP", "portal RH", "CRM", "e-mail corporativo", "VPN"], "hardware": ["notebook", "monitor", "teclado", "headset", "dock"],
             "software": ["Excel", "Power BI", "Teams", "VS Code", "Outlook"], "rede": ["3º andar", "escritório de Lisboa", "auditório", "sala de reuniões", "home office"],
             "financeiro": ["150", "480", "1200", "75", "320"]}
    urgencia = {"alta": ["Isso está parando toda a equipe!", "É urgente, tenho uma entrega hoje."],
                "media": ["Preciso disso resolvido esta semana.", "Está atrapalhando meu trabalho."],
                "baixa": ["Sem pressa.", "Quando puderem, por favor."]}
    saudacoes = ["Olá,", "Bom dia!", "Oi pessoal,", "Prezados,", ""]
    nomes = ["ana.souza", "joao.lima", "carla.mendes", "pedro.alves", "rita.costa"]
    linhas = []
    for _ in range(n):
        cat = rnd.choice(CATEGORIAS); texto, resumo = rnd.choice(modelos[cat]); x = rnd.choice(slots[cat]); prio = rnd.choice(PRIORIDADES)
        corpo = f"{rnd.choice(saudacoes)} {texto.format(x=x)} {rnd.choice(urgencia[prio])}"
        if rnd.random() < 0.5: corpo += f"  Att, {rnd.choice(nomes)}@empresa.com, tel (11) 9{rnd.randint(1000, 9999)}-{rnd.randint(1000, 9999)}"
        if rnd.random() < 0.2: corpo = "   " + corpo.replace(" ", "  ", 2)
        rot_cat = rnd.choice([cat, cat, cat.upper(), f" {cat} "]); rot_prio = rnd.choice([prio, prio, prio.capitalize(), "média" if prio == "media" else prio])
        linhas.append({"texto": corpo, "categoria": rot_cat, "prioridade": rot_prio, "resumo": resumo.format(x=x)})
    linhas += [dict(rnd.choice(linhas)) for _ in range(n // 12)]                      # duplicatas exatas
    linhas += [{"texto": "", "categoria": "acesso", "prioridade": "baixa", "resumo": ""} for _ in range(n // 40)]   # vazios
    rnd.shuffle(linhas); return linhas

# --- Pipeline de preparação (construído passo a passo na Semana 17) ---
import re, json, unicodedata
def preparar_tickets(n=600, seed=0):
    """Devolve (treino, val, teste): listas de {"messages": [system, user, assistant]}."""
    norm = lambda s: unicodedata.normalize("NFKD", s.strip().lower()).encode("ascii", "ignore").decode()
    vistos, exemplos = set(), []
    for l in gerar_tickets_brutos(n, seed):
        t = re.sub(r"\s+", " ", l["texto"]).strip()
        t = re.sub(r"[\w.]+@[\w.]+", "<EMAIL>", t); t = re.sub(r"\(\d{2}\)\s*9?\d{4}-\d{4}", "<TELEFONE>", t)
        if not t or t in vistos: continue
        vistos.add(t)
        alvo = {"categoria": norm(l["categoria"]), "prioridade": norm(l["prioridade"]), "resumo": l["resumo"]}
        exemplos.append({"messages": [{"role": "system", "content": SISTEMA_TICKETS}, {"role": "user", "content": t},
                                      {"role": "assistant", "content": json.dumps(alvo, ensure_ascii=False)}]})
    random.Random(seed).shuffle(exemplos); a, b = int(0.8 * len(exemplos)), int(0.9 * len(exemplos))
    return exemplos[:a], exemplos[a:b], exemplos[b:]

treino, val, teste = preparar_tickets()
print(len(treino), len(val), len(teste)); print(json.dumps(treino[0], ensure_ascii=False, indent=1))

## Passo 2 — Base model e generation
Carregamos em fp32 e usamos mixed precision no treino.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO)
base = AutoModelForCausalLM.from_pretrained(MODELO).to(device).eval()
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

def gerar(modelo, mensagens, max_new_tokens=64):
    prompt = tok.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    ent = tok(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    with torch.no_grad():
        out = modelo.generate(**ent, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=PAD)
    return tok.decode(out[0, ent["input_ids"].shape[1]:], skip_special_tokens=True).strip()

ex = val[0]; print("ENTRADA :", ex["messages"][1]["content"]); print("ESPERADO:", ex["messages"][2]["content"]); print("BASE    :", gerar(base, ex["messages"][:2]))

## Passo 3 — Avaliação da tarefa (antes do treino)
Para cada exemplo: gere, tente extrair e interpretar o JSON, compare com o gabarito.

In [ ]:
def avaliar(modelo, dados, n=40):
    linhas = []
    for e in dados[:n]:
        ouro = json.loads(e["messages"][-1]["content"]); bruto = gerar(modelo, e["messages"][:-1])
        # [SEU CÓDIGO AQUI] extraia o trecho entre a primeira "{" e a última "}" e interprete; defina pred e valido
        linhas.append({"json_valido": valido, "categoria_ok": pred.get("categoria") == ouro["categoria"],
                       "prioridade_ok": pred.get("prioridade") == ouro["prioridade"],
                       "tudo_ok": valido and all(pred.get(k) == ouro[k] for k in ouro)})
    return pd.DataFrame(linhas).mean()

antes = avaliar(base, val); print(antes.round(3))

## Passo 4 — Codificar um exemplo com mascaramento
`codificar` devolve `(input_ids, labels)`:
1. `prompt` = template aplicado às mensagens **sem** a do assistente, com `add_generation_prompt=True`.
2. `ids_p` = tokens do prompt; `ids_r` = tokens de `resposta + tok.eos_token`. Use `add_special_tokens=False`.
3. `labels` = `-100` para cada token do prompt, seguido de `ids_r`.

In [ ]:
def codificar(mensagens, max_len=256):
    # [SEU CÓDIGO AQUI] monte ids e labels
    return ids[:max_len], labels[:max_len]

ids, labels = codificar(treino[0]["messages"])
print("tokens:", len(ids), "| posições que contam na perda:", sum(l != -100 for l in labels))
print("o modelo aprende a escrever:", repr(tok.decode([l for l in labels if l != -100])))

## Passo 5 — Montar o batch
Padding à direita até o maior exemplo do batch: `PAD` em `input_ids`, `0` em `attention_mask`, `-100` em `labels`.

In [ ]:
def colar(exemplos):
    cods = [codificar(e["messages"]) for e in exemplos]; T = max(len(i) for i, _ in cods)
    # [SEU CÓDIGO AQUI] crie os três tensores com padding
    return {"input_ids": input_ids.to(device), "attention_mask": attention_mask.to(device), "labels": labels.to(device)}

lote = colar(treino[:4]); print({k: tuple(v.shape) for k, v in lote.items()})

## Passo 6 — Aplicar LoRA
Descubra os nomes dos módulos lineares de attention e configure o adapter (`r=16`, `lora_alpha=32`, `lora_dropout=0.05`).

In [ ]:
print(sorted({n.split(".")[-1] for n, m in base.named_modules() if isinstance(m, torch.nn.Linear) or type(m).__name__ == "Conv1D"}))
# [SEU CÓDIGO AQUI] crie config (LoraConfig) e modelo (get_peft_model)
modelo.print_trainable_parameters()

## Passo 7 — Treinar
Loop manual com mixed precision. Só os parâmetros com `requires_grad=True` vão para o optimizer.

In [ ]:
@torch.no_grad()
def perda_val():
    modelo.eval(); tot = 0.0
    for i in range(0, len(val), 8):
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
            tot += modelo(**colar(val[i:i + 8])).loss.item() * len(val[i:i + 8])
    modelo.train(); return tot / len(val)

EPOCAS, LOTE = 2, 8
# [SEU CÓDIGO AQUI] liste os parâmetros treináveis e crie o AdamW (lr=2e-4)
scaler = torch.amp.GradScaler(enabled=device == "cuda"); hist, hist_val = [], [(0, perda_val())]; rnd = random.Random(0); modelo.train()
for epoca in range(EPOCAS):
    ordem = list(range(len(treino))); rnd.shuffle(ordem)
    for n, i in enumerate(range(0, len(ordem), LOTE)):
        lote = colar([treino[j] for j in ordem[i:i + LOTE]])
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
            # [SEU CÓDIGO AQUI] forward -> loss
        otimizador.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(otimizador)
        torch.nn.utils.clip_grad_norm_(treinaveis, 1.0); scaler.step(otimizador); scaler.update(); hist.append(loss.item())
        if n % 20 == 0: print(f"época {epoca} passo {n:3d} | loss {loss.item():.3f}")
    hist_val.append((len(hist), perda_val())); print(f"== época {epoca} | perda de validação {hist_val[-1][1]:.3f}")
modelo.eval()
plt.plot(hist, label="treino"); plt.plot(*zip(*hist_val), "o-", label="validação"); plt.yscale("log"); plt.legend(); plt.show()

## Passo 8 — Depois do treino

In [ ]:
print("ENTRADA :", ex["messages"][1]["content"]); print("ESPERADO:", ex["messages"][2]["content"]); print("AJUSTADO:", gerar(modelo, ex["messages"][:2]))
depois = avaliar(modelo, val)
pd.DataFrame({"antes": antes, "depois": depois}).round(3)

## Passo 9 — Salvar e recarregar o adapter
Salve só o adapter, veja o tamanho em disco, recarregue sobre um base model novo e confirme a mesma saída.

In [ ]:
# [SEU CÓDIGO AQUI] salve o adapter na pasta "adaptador-chamados"
mb = sum(os.path.getsize(os.path.join("adaptador-chamados", f)) for f in os.listdir("adaptador-chamados")) / 1e6
print(f"adaptador: {mb:.1f} MB | arquivos: {os.listdir('adaptador-chamados')}")

base2 = AutoModelForCausalLM.from_pretrained(MODELO).to(device)
# [SEU CÓDIGO AQUI] aplique o adapter salvo sobre base2
assert gerar(recarregado, ex["messages"][:2]) == gerar(modelo, ex["messages"][:2]); print("adaptador recarregado com a mesma saída")

## Passo 10 — O modelo aprendeu a parar?
Gere com um limite alto de tokens. Se a resposta termina logo após o `}`, o token de fim foi aprendido.

In [ ]:
r = gerar(modelo, teste[0]["messages"][:2], max_new_tokens=200); print(len(tok(r)["input_ids"]), "tokens gerados:\n", r)

**Reflexão:** remova `+ tok.eos_token` em `codificar`, treine de novo e repita esta célula. O que muda?